# 07b — PCA Dislocation and Macro-Enhanced Momentum Dashboard

Notebook 07 is the unsupervised representation learning baseline for the canonical 29-asset universe. It explains correlation structure, PCA factors, eigenportfolio diagnostics, clustering, rolling PCA stability, and existing macro-regime overlays without making allocation or trading claims.

Notebook 07b extends that baseline into a tactical diagnostic dashboard. It asks which assets are currently rich or cheap versus a rolling PCA reconstruction, and whether trailing momentum and existing macro/regime context confirm or contradict that dislocation.

```text
07  = unsupervised representation learning baseline
07b = PCA dislocation + momentum + macro/regime diagnostic dashboard
```

This notebook is inspired by external practitioner research on cross-asset PCA dislocations, momentum confirmation, and macro-aware diagnostics, but it is adapted to this repo's 29-asset panel and existing regime labels. It is not a strategy, not an allocation engine, not a proprietary-research replication, not a backtest, and not a live trading signal.

---

**Paper 2 — tactical diagnostic dashboard:** 07b is the tactical extension of the [07](07_unsupervised_representation_learning.ipynb) structural layer. Both are **off-axis diagnostic chapters** — they are distinct from the ML (03), DL (04/05/05b), and RL (06a/06b/06c) performance legs and make no out-of-sample allocation claim. The scope disclaimer in the final paragraph above applies to the full notebook; see [00_overview.ipynb](00_overview.ipynb) for the reading guide.

## §0 Setup and relationship to Notebook 07

This section sets deterministic paths, imports the canonical universe metadata, and creates the Notebook 07b artifact directories. All outputs from this notebook are contained under `results/notebook_07b/`.

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

SEED = 42
np.random.seed(SEED)

TRADING_DAYS = 252
WEEKS_PER_YEAR = 52
ROLLING_WINDOW_WEEKS = 260
PCA_COMPONENTS = 5
RESIDUAL_Z_LOOKBACK = 156
RESIDUAL_Z_MIN_PERIODS = 52
MOMENTUM_HORIZONS = [4, 13, 26, 52]
TOP_N_TS_ASSETS = 5

ROOT = Path.cwd().resolve()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent.resolve()

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from aiam.data.universe import UNIVERSE_29
from aiam.features.asset_class import ASSET_CLASS_MAP

OUT_DIR = ROOT / "results" / "notebook_07b"
FIG_DIR = OUT_DIR / "figures"
HTML_DIR = OUT_DIR / "html"
for path in [OUT_DIR, FIG_DIR, HTML_DIR]:
    path.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.figsize": (10, 6),
    "axes.grid": True,
    "grid.alpha": 0.2,
    "axes.spines.top": False,
    "axes.spines.right": False,
})
import matplotlib
matplotlib.rcParams.update({
    "font.family": "serif",
    "font.size": 10,
    "figure.dpi": 120,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
})

print(f"ROOT: {ROOT}")
print(f"Python: {sys.version.split()[0]}")
print(f"Output directory: {OUT_DIR.relative_to(ROOT)}")
print(f"Notebook role: diagnostic dashboard, not a strategy or allocation engine")

## §1 Load canonical 29-asset panel and relevant artifacts

This section loads the canonical daily price panel, daily return panel, static asset metadata, and the existing monthly `dominant_regime` labels when available. The return panel is used as a consistency input, while weekly returns for the PCA dislocation analysis are computed directly from daily prices.

In [ ]:
prices_path = ROOT / "data" / "cache" / "prices_29.parquet"
returns_path = ROOT / "data" / "cache" / "returns_29_2003_2026.parquet"
regime_cache_path = ROOT / "data" / "cache" / "regime_signals_2003_2026.parquet"
regime_published_path = ROOT / "data" / "published" / "regime_signals.parquet"
notebook07_dir = ROOT / "results" / "notebook_07"

if not prices_path.exists():
    raise FileNotFoundError(f"Required canonical prices missing: {prices_path}")

prices = pd.read_parquet(prices_path)
prices.index = pd.to_datetime(prices.index)
prices = prices.sort_index().reindex(columns=UNIVERSE_29)

if returns_path.exists():
    daily_returns = pd.read_parquet(returns_path)
    daily_returns.index = pd.to_datetime(daily_returns.index)
    daily_returns = daily_returns.sort_index().reindex(columns=UNIVERSE_29)
    returns_source = returns_path
else:
    daily_returns = prices.pct_change(fill_method=None)
    returns_source = prices_path

if regime_cache_path.exists():
    regime_path = regime_cache_path
elif regime_published_path.exists():
    regime_path = regime_published_path
else:
    regime_path = None

regimes = None
if regime_path is not None:
    regimes = pd.read_parquet(regime_path)
    regimes.index = pd.to_datetime(regimes.index)
    regimes = regimes.sort_index()

asset_meta = pd.DataFrame({
    "asset": UNIVERSE_29,
    "asset_class": [ASSET_CLASS_MAP.get(asset, "unknown") for asset in UNIVERSE_29],
}).set_index("asset")

assert prices.shape[1] == 29, f"Expected 29 price columns, got {prices.shape[1]}"
assert daily_returns.shape[1] == 29, f"Expected 29 return columns, got {daily_returns.shape[1]}"
assert list(prices.columns) == list(UNIVERSE_29), "Price columns must match UNIVERSE_29"
assert list(daily_returns.columns) == list(UNIVERSE_29), "Return columns must match UNIVERSE_29"

def inventory_date_range(obj):
    if obj is None:
        return "", ""
    idx = obj.index
    if isinstance(idx, pd.DatetimeIndex):
        return str(idx.min().date()), str(idx.max().date())
    return "", ""

inventory_rows = []
for label, path, obj, role, frequency in [
    ("canonical_daily_prices", prices_path, prices, "source for weekly returns and momentum", "daily"),
    ("canonical_daily_returns", returns_source, daily_returns, "sanity check and fallback return panel", "daily"),
    ("asset_metadata", ROOT / "src" / "aiam" / "features" / "asset_class.py", asset_meta, "asset-class grouping", "static"),
    ("notebook_07_artifacts", notebook07_dir, None, "optional provenance/context", "static"),
]:
    start, end = inventory_date_range(obj)
    inventory_rows.append({
        "input": label,
        "path": str(path.relative_to(ROOT)) if path.exists() else str(path),
        "exists": bool(path.exists()),
        "shape": "" if obj is None else f"{obj.shape[0]}x{obj.shape[1]}",
        "start": start,
        "end": end,
        "frequency": frequency,
        "role": role,
    })

if regime_path is not None:
    inventory_rows.append({
        "input": "macro_regime_labels",
        "path": str(regime_path.relative_to(ROOT)),
        "exists": True,
        "shape": f"{regimes.shape[0]}x{regimes.shape[1]}",
        "start": str(regimes.index.min().date()),
        "end": str(regimes.index.max().date()),
        "frequency": "monthly",
        "role": "dominant_regime context overlay",
    })
else:
    inventory_rows.append({
        "input": "macro_regime_labels",
        "path": "",
        "exists": False,
        "shape": "",
        "start": "",
        "end": "",
        "frequency": "monthly",
        "role": "skipped; unavailable",
    })

input_inventory = pd.DataFrame(inventory_rows)
input_inventory.to_csv(OUT_DIR / "input_inventory.csv", index=False)

print(f"Prices shape: {prices.shape}")
print(f"Returns shape: {daily_returns.shape}")
print(f"Daily date range: {prices.index.min().date()} to {prices.index.max().date()}")
print(f"Asset count: {len(UNIVERSE_29)}")
print(f"Asset classes: {sorted(asset_meta['asset_class'].unique())}")
print(f"Regime file path used: {regime_path.relative_to(ROOT) if regime_path else 'unavailable'}")
input_inventory

## §2 Weekly return panel and rolling PCA design

The PCA dislocation model uses weekly returns computed from canonical daily prices. Version 1 uses complete-case weekly history across all 29 assets, preserving the full asset universe while making the effective sample start date explicit.

Rolling PCA choices are locked for this pass:

- Weekly returns from daily prices.
- Complete-case 29-asset weekly history.
- Five-year rolling window, approximated as 260 weekly observations.
- Fixed `K=5` principal components.
- Standardization fitted inside each rolling window only.

In [ ]:
weekly_prices_raw = prices.resample("W-FRI").last()
weekly_returns_raw = weekly_prices_raw.pct_change(fill_method=None)
weekly_returns = weekly_returns_raw.dropna(axis=0, how="any").copy()
weekly_prices = weekly_prices_raw.loc[weekly_returns.index, UNIVERSE_29].copy()

assert weekly_returns.shape[1] == 29
assert list(weekly_returns.columns) == list(UNIVERSE_29)
assert weekly_returns.index.is_monotonic_increasing

coverage = pd.DataFrame({
    "asset": UNIVERSE_29,
    "asset_class": asset_meta.loc[UNIVERSE_29, "asset_class"].values,
    "first_valid_weekly_price": [weekly_prices_raw[a].first_valid_index() for a in UNIVERSE_29],
    "first_complete_case_weekly_return": weekly_returns.index.min(),
    "last_complete_case_weekly_return": weekly_returns.index.max(),
    "complete_case_weekly_observations": len(weekly_returns),
})
coverage["first_valid_weekly_price"] = pd.to_datetime(coverage["first_valid_weekly_price"]).dt.date.astype(str)
coverage["first_complete_case_weekly_return"] = pd.to_datetime(coverage["first_complete_case_weekly_return"]).dt.date.astype(str)
coverage["last_complete_case_weekly_return"] = pd.to_datetime(coverage["last_complete_case_weekly_return"]).dt.date.astype(str)

print(f"Weekly return shape: {weekly_returns.shape}")
print(f"Weekly complete-case date range: {weekly_returns.index.min().date()} to {weekly_returns.index.max().date()}")
print(f"Number of assets: {weekly_returns.shape[1]}")
print(f"Rolling window length: {ROLLING_WINDOW_WEEKS} weekly observations")
print(f"PCA components K: {PCA_COMPONENTS}")
coverage.head(10)

## §3 Rolling PCA reconstruction residuals

For each weekly window end date, the notebook standardizes returns inside that historical window, fits PCA with five components, reconstructs the latest standardized return vector, and stores the residual. The residual is the observed standardized return minus the PCA reconstruction.

Positive residuals mean the asset was stronger than the rolling PCA reconstruction for that week. Negative residuals mean the asset was weaker than the rolling PCA reconstruction for that week.

In [ ]:
residual_rows = []
diagnostic_rows = []

for end_pos in range(ROLLING_WINDOW_WEEKS, len(weekly_returns) + 1):
    window_returns = weekly_returns.iloc[end_pos - ROLLING_WINDOW_WEEKS:end_pos]
    window_end = window_returns.index[-1]
    window_start = window_returns.index[0]

    means = window_returns.mean()
    stds = window_returns.std(ddof=1)
    if stds.le(0).any():
        continue

    x = (window_returns - means) / stds
    pca = PCA(n_components=PCA_COMPONENTS, random_state=SEED)
    pca.fit(x)

    latest_x = x.iloc[[-1]]
    scores = pca.transform(latest_x)
    reconstructed = scores @ pca.components_ + pca.mean_
    residual = latest_x.to_numpy().ravel() - reconstructed.ravel()

    residual_rows.append(pd.Series(residual, index=UNIVERSE_29, name=window_end))

    reconstruction_rmse = float(np.sqrt(np.mean(residual ** 2)))
    reconstruction_mae = float(np.mean(np.abs(residual)))
    explained = pca.explained_variance_ratio_
    diagnostic_rows.append({
        "date": window_end,
        "window_start": window_start,
        "window_end": window_end,
        "observations": len(window_returns),
        "n_assets": len(UNIVERSE_29),
        "n_components": PCA_COMPONENTS,
        "pc1_explained_variance_ratio": explained[0],
        "pc2_explained_variance_ratio": explained[1],
        "pc3_explained_variance_ratio": explained[2],
        "pc4_explained_variance_ratio": explained[3],
        "pc5_explained_variance_ratio": explained[4],
        "pc1_pc5_cumulative_explained_variance_ratio": float(explained.sum()),
        "reconstruction_rmse": reconstruction_rmse,
        "reconstruction_mae": reconstruction_mae,
    })

pca_residuals = pd.DataFrame(residual_rows)
pca_residuals.index.name = "date"
pca_residuals = pca_residuals.reindex(columns=UNIVERSE_29)
rolling_pca_diagnostics = pd.DataFrame(diagnostic_rows)

pca_residuals.to_parquet(OUT_DIR / "pca_reconstruction_residuals.parquet")
rolling_pca_diagnostics.to_csv(OUT_DIR / "rolling_pca_reconstruction_diagnostics.csv", index=False)

print(f"Residual matrix shape: {pca_residuals.shape}")
print(f"Residual date range: {pca_residuals.index.min().date()} to {pca_residuals.index.max().date()}")
print(f"Rolling diagnostics rows: {len(rolling_pca_diagnostics)}")
rolling_pca_diagnostics.tail()

## §4 PCA dislocation scores

This section converts PCA reconstruction residuals into trailing residual z-scores. The score uses only residual history available through each date. The sign convention is:

- Positive `pca_dislocation_z`: richer / stronger than PCA reconstruction.
- Negative `pca_dislocation_z`: cheaper / weaker than PCA reconstruction.

The z-score is a diagnostic measure of stretch versus the rolling PCA reconstruction. It is not a buy/sell signal.

In [ ]:
trailing_residual_std = pca_residuals.rolling(
    RESIDUAL_Z_LOOKBACK,
    min_periods=RESIDUAL_Z_MIN_PERIODS,
).std(ddof=1)

dislocation_z = pca_residuals / trailing_residual_std.replace(0, np.nan)

def dislocation_bucket(z: float) -> str:
    if pd.isna(z):
        return "unavailable"
    if z >= 1.0:
        return "rich"
    if z <= -1.0:
        return "cheap"
    return "neutral"

score_frames = []
for date, row in dislocation_z.iterrows():
    frame = pd.DataFrame({
        "date": date,
        "asset": UNIVERSE_29,
        "pca_residual": pca_residuals.loc[date, UNIVERSE_29].values,
        "pca_dislocation_z": row.reindex(UNIVERSE_29).values,
    })
    frame["dislocation_bucket"] = frame["pca_dislocation_z"].map(dislocation_bucket)
    frame["abs_dislocation_rank"] = frame["pca_dislocation_z"].abs().rank(ascending=False, method="first")
    score_frames.append(frame)

pca_dislocation_scores = pd.concat(score_frames, ignore_index=True)
pca_dislocation_scores["date"] = pd.to_datetime(pca_dislocation_scores["date"])
pca_dislocation_scores.to_csv(OUT_DIR / "pca_dislocation_scores.csv", index=False)

latest_dislocation_date = pca_dislocation_scores.dropna(subset=["pca_dislocation_z"])["date"].max()
latest_dislocations = (
    pca_dislocation_scores[pca_dislocation_scores["date"] == latest_dislocation_date]
    .merge(asset_meta.reset_index(), on="asset", how="left")
    .sort_values("abs_dislocation_rank")
)

print(f"Dislocation score rows: {len(pca_dislocation_scores):,}")
print(f"Latest dislocation score date: {latest_dislocation_date.date()}")
latest_dislocations.head(10)

## §5 Momentum diagnostics

Momentum is trailing-only and used as confirmation or contradiction context. The notebook computes 4-week, 13-week, 26-week, and 52-week returns from weekly prices. Each horizon is z-scored cross-sectionally at each date, and the composite `momentum_score` is the average of available horizon z-scores.

In [ ]:
def cross_sectional_zscore(frame: pd.DataFrame) -> pd.DataFrame:
    row_mean = frame.mean(axis=1)
    row_std = frame.std(axis=1, ddof=1).replace(0, np.nan)
    return frame.sub(row_mean, axis=0).div(row_std, axis=0)

momentum_wide = {}
for horizon in MOMENTUM_HORIZONS:
    momentum_wide[f"mom_{horizon}w"] = weekly_prices.pct_change(horizon, fill_method=None)

momentum_z_wide = {f"mom_{horizon}w_z": cross_sectional_zscore(momentum_wide[f"mom_{horizon}w"]) for horizon in MOMENTUM_HORIZONS}

momentum_score = pd.concat(momentum_z_wide.values(), axis=0).groupby(level=0).mean().reindex(columns=UNIVERSE_29)

momentum_frames = []
for date in weekly_returns.index:
    frame = pd.DataFrame({"date": date, "asset": UNIVERSE_29})
    for horizon in MOMENTUM_HORIZONS:
        raw_name = f"mom_{horizon}w"
        z_name = f"mom_{horizon}w_z"
        frame[raw_name] = momentum_wide[raw_name].reindex(index=[date], columns=UNIVERSE_29).iloc[0].values
        frame[z_name] = momentum_z_wide[z_name].reindex(index=[date], columns=UNIVERSE_29).iloc[0].values
    frame["momentum_score"] = momentum_score.reindex(index=[date], columns=UNIVERSE_29).iloc[0].values
    momentum_frames.append(frame)

momentum_scores = pd.concat(momentum_frames, ignore_index=True)

def momentum_bucket(score: float) -> str:
    if pd.isna(score):
        return "unavailable"
    if score >= 0.5:
        return "positive_momentum"
    if score <= -0.5:
        return "negative_momentum"
    return "neutral_momentum"

momentum_scores["momentum_bucket"] = momentum_scores["momentum_score"].map(momentum_bucket)
momentum_scores["date"] = pd.to_datetime(momentum_scores["date"])
momentum_scores.to_csv(OUT_DIR / "momentum_scores.csv", index=False)

latest_momentum = (
    momentum_scores[momentum_scores["date"] == latest_dislocation_date]
    .merge(asset_meta.reset_index(), on="asset", how="left")
    .sort_values("momentum_score", ascending=False)
)

print(f"Momentum score rows: {len(momentum_scores):,}")
print(f"Latest momentum score date: {latest_dislocation_date.date()}")
latest_momentum.head(10)

## §6 Macro/regime overlay

This section aligns the existing monthly `dominant_regime` labels to weekly dates. Alignment uses only labels available at or before each weekly date by forward-filling from the regime observation date. The regime label is categorical context and color only; it is not a regime-switching rule or macro support score.

In [ ]:
if regimes is not None and "dominant_regime" in regimes.columns:
    regime_series = regimes["dominant_regime"].dropna().sort_index()
    aligned_regime = regime_series.reindex(weekly_returns.index, method="ffill")
    macro_regime_overlay = pd.DataFrame({
        "date": weekly_returns.index,
        "dominant_regime": aligned_regime.values,
    })
    macro_regime_overlay["macro_context"] = np.where(
        macro_regime_overlay["dominant_regime"].isna(),
        "regime_unavailable",
        "regime_" + macro_regime_overlay["dominant_regime"].astype("Int64").astype(str),
    )
    regime_source = regime_path.relative_to(ROOT)
else:
    macro_regime_overlay = pd.DataFrame({
        "date": weekly_returns.index,
        "dominant_regime": pd.Series([pd.NA] * len(weekly_returns), dtype="Int64"),
        "macro_context": "regime_unavailable",
    })
    regime_source = "unavailable"

macro_regime_overlay["date"] = pd.to_datetime(macro_regime_overlay["date"])
macro_regime_overlay.to_csv(OUT_DIR / "macro_regime_overlay.csv", index=False)

print(f"Regime source: {regime_source}")
print(f"Aligned regime rows: {len(macro_regime_overlay):,}")
print(f"Missing aligned regime labels: {macro_regime_overlay['dominant_regime'].isna().sum():,}")
macro_regime_overlay.tail()

## §7 Cross-sectional scorecard

The scorecard joins the latest PCA dislocation scores, trailing momentum diagnostics, asset metadata, and aligned regime label. Diagnostic labels classify whether momentum confirms or contradicts the latest rich/cheap dislocation, without creating recommendations or portfolio actions.

In [ ]:
latest_date = latest_dislocation_date
latest_dislocation_table = pca_dislocation_scores[pca_dislocation_scores["date"] == latest_date].copy()
latest_momentum_table = momentum_scores[momentum_scores["date"] == latest_date].copy()
latest_regime_table = macro_regime_overlay[macro_regime_overlay["date"] == latest_date].copy()

scorecard = (
    latest_dislocation_table
    .merge(latest_momentum_table[["date", "asset", "momentum_score", "momentum_bucket"]], on=["date", "asset"], how="left")
    .merge(asset_meta.reset_index(), on="asset", how="left")
)

if latest_regime_table.empty:
    scorecard["dominant_regime"] = pd.NA
    scorecard["macro_context"] = "regime_unavailable"
else:
    scorecard["dominant_regime"] = latest_regime_table["dominant_regime"].iloc[0]
    scorecard["macro_context"] = latest_regime_table["macro_context"].iloc[0]

def diagnostic_label(row: pd.Series) -> str:
    if row["dislocation_bucket"] == "rich" and row["momentum_bucket"] == "positive_momentum":
        return "rich_with_positive_momentum"
    if row["dislocation_bucket"] == "rich" and row["momentum_bucket"] == "negative_momentum":
        return "rich_with_negative_momentum"
    if row["dislocation_bucket"] == "cheap" and row["momentum_bucket"] == "positive_momentum":
        return "cheap_with_positive_momentum"
    if row["dislocation_bucket"] == "cheap" and row["momentum_bucket"] == "negative_momentum":
        return "cheap_with_negative_momentum"
    return "neutral_or_mixed"

scorecard["diagnostic_label"] = scorecard.apply(diagnostic_label, axis=1)
scorecard = scorecard[[
    "date",
    "asset",
    "asset_class",
    "pca_dislocation_z",
    "dislocation_bucket",
    "momentum_score",
    "momentum_bucket",
    "dominant_regime",
    "macro_context",
    "diagnostic_label",
    "abs_dislocation_rank",
]].sort_values("abs_dislocation_rank")

scorecard.to_csv(OUT_DIR / "pca_dislocation_scorecard.csv", index=False)

methodology_gap_table = pd.DataFrame([
    {
        "methodology_element": "True institutional positioning layer",
        "availability_in_repo": "unavailable",
        "07b_v1_treatment": "not implemented; no positioning proxy is fabricated",
    },
    {
        "methodology_element": "Futures non-commercial positioning / fund beta / survey / option positioning",
        "availability_in_repo": "unavailable",
        "07b_v1_treatment": "explicitly excluded from v1",
    },
    {
        "methodology_element": "Point-in-time macro support scores",
        "availability_in_repo": "unavailable",
        "07b_v1_treatment": "replaced only by existing dominant_regime categorical context",
    },
    {
        "methodology_element": "Broad institutional cross-asset index universe",
        "availability_in_repo": "unavailable",
        "07b_v1_treatment": "uses repo-native canonical 29-asset panel",
    },
    {
        "methodology_element": "OU residual model and integrated residuals",
        "availability_in_repo": "not implemented",
        "07b_v1_treatment": "simple trailing residual z-score",
    },
    {
        "methodology_element": "Lasso PC relevance selection",
        "availability_in_repo": "not implemented",
        "07b_v1_treatment": "fixed K=5 PCA reconstruction",
    },
])
methodology_gap_table.to_csv(OUT_DIR / "methodology_gap_table.csv", index=False)

print(f"Latest scorecard date: {latest_date.date()}")
print(f"Scorecard rows: {len(scorecard)}")
scorecard.head(12)

## §8 Dashboard-style visualizations

This section saves static Matplotlib dashboard figures. These views are intended for inspection and review: rolling PCA diagnostics, latest rich/cheap dislocations, dislocation-versus-momentum context, regime-aware scorecard heatmap, and the top five latest absolute dislocation time series.

In [ ]:
# 1. Rolling PCA diagnostics.
diag = rolling_pca_diagnostics.copy()
diag["date"] = pd.to_datetime(diag["date"])

fig, axes = plt.subplots(2, 1, figsize=(11, 8), sharex=True)
axes[0].plot(diag["date"], diag["pc1_pc5_cumulative_explained_variance_ratio"], color="#2F5597", label="PC1-PC5 cumulative explained variance")
axes[0].set_ylabel("Cumulative explained variance")
axes[0].set_title("Rolling PCA Explained Variance, Fixed K=5")
axes[0].legend(loc="best")
axes[1].plot(diag["date"], diag["reconstruction_rmse"], color="#A23E48", label="Reconstruction RMSE")
axes[1].set_ylabel("Standardized residual RMSE")
axes[1].set_xlabel("Weekly window end date")
axes[1].set_title("Rolling PCA Reconstruction Error")
axes[1].legend(loc="best")
fig.tight_layout()
fig.savefig(FIG_DIR / "rolling_pca_reconstruction_diagnostics.png", dpi=150, bbox_inches="tight")
plt.show()

# 2. Latest dislocation bar chart.
bar_data = scorecard.sort_values("pca_dislocation_z")
bar_colors = np.where(bar_data["pca_dislocation_z"] >= 0, "#4C78A8", "#E45756")
fig, ax = plt.subplots(figsize=(12, 7))
ax.bar(bar_data["asset"], bar_data["pca_dislocation_z"], color=bar_colors)
ax.axhline(0, color="black", linewidth=0.8)
ax.axhline(1, color="#4C78A8", linewidth=0.8, linestyle="--")
ax.axhline(-1, color="#E45756", linewidth=0.8, linestyle="--")
ax.set_title(f"Latest PCA Dislocation Z-Scores ({latest_date.date()})")
ax.set_xlabel("Asset")
ax.set_ylabel("PCA dislocation z-score\npositive = rich/strong, negative = cheap/weak")
ax.tick_params(axis="x", rotation=90)
fig.tight_layout()
fig.savefig(FIG_DIR / "latest_dislocation_bar.png", dpi=150, bbox_inches="tight")
plt.show()

# 3. Dislocation vs momentum scatter.
classes = scorecard["asset_class"].fillna("unknown").unique().tolist()
color_map = dict(zip(classes, plt.cm.tab10(np.linspace(0, 1, len(classes)))))
fig, ax = plt.subplots(figsize=(10, 7))
for asset_class, group in scorecard.groupby("asset_class", dropna=False):
    ax.scatter(
        group["pca_dislocation_z"],
        group["momentum_score"],
        label=asset_class,
        s=70,
        alpha=0.85,
        color=color_map.get(asset_class, "gray"),
    )
    for _, row in group.iterrows():
        ax.annotate(row["asset"].replace(".US", ""), (row["pca_dislocation_z"], row["momentum_score"]), fontsize=8, xytext=(4, 4), textcoords="offset points")
ax.axvline(0, color="black", linewidth=0.8)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xlabel("PCA dislocation z-score")
ax.set_ylabel("Composite momentum score")
ax.set_title(f"PCA Dislocation vs Momentum ({latest_date.date()})")
ax.legend(loc="center left", bbox_to_anchor=(1, 0.5), fontsize=8)
fig.tight_layout()
fig.savefig(FIG_DIR / "dislocation_vs_momentum_scatter.png", dpi=150, bbox_inches="tight")
plt.show()

# 4. Macro/regime scorecard heatmap.
heatmap_data = scorecard.set_index("asset")[["pca_dislocation_z", "momentum_score"]].copy()
heatmap_data = heatmap_data.loc[scorecard.sort_values(["asset_class", "abs_dislocation_rank"])["asset"]]
fig, ax = plt.subplots(figsize=(7, 10))
im = ax.imshow(heatmap_data.values, aspect="auto", cmap="coolwarm", vmin=-2.5, vmax=2.5)
ax.set_xticks(range(heatmap_data.shape[1]))
ax.set_xticklabels(["PCA dislocation z", "Momentum score"], rotation=30, ha="right")
ax.set_yticks(range(heatmap_data.shape[0]))
ax.set_yticklabels(heatmap_data.index)
regime_label = scorecard["macro_context"].iloc[0]
ax.set_title(f"Scorecard Heatmap by Asset ({latest_date.date()}, {regime_label})")
for i in range(heatmap_data.shape[0]):
    for j in range(heatmap_data.shape[1]):
        val = heatmap_data.iloc[i, j]
        ax.text(j, i, "" if pd.isna(val) else f"{val:.1f}", ha="center", va="center", fontsize=7, color="black")
fig.colorbar(im, ax=ax, label="z-score / standardized score")
fig.tight_layout()
fig.savefig(FIG_DIR / "macro_regime_scorecard_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

# 5. Selected asset dislocation time series.
top_assets = scorecard.sort_values("abs_dislocation_rank").head(TOP_N_TS_ASSETS)["asset"].tolist()
dislocation_z_ts = dislocation_z.reindex(columns=top_assets)
fig, ax = plt.subplots(figsize=(11, 6))
for asset in top_assets:
    ax.plot(dislocation_z_ts.index, dislocation_z_ts[asset], label=asset)
for level in [-2, -1, 0, 1, 2]:
    ax.axhline(level, color="black" if level == 0 else "gray", linewidth=0.8, linestyle="-" if level == 0 else "--", alpha=0.7)
ax.set_title("Top 5 Latest Absolute PCA Dislocations Through Time")
ax.set_xlabel("Weekly date")
ax.set_ylabel("PCA dislocation z-score")
ax.legend(loc="best", fontsize=8)
fig.tight_layout()
fig.savefig(FIG_DIR / "selected_asset_dislocation_timeseries.png", dpi=150, bbox_inches="tight")
plt.show()

print("Saved dashboard figures:")
for path in sorted(FIG_DIR.glob("*.png")):
    print(f"- {path.relative_to(ROOT)}")

## §9 Methodology gaps versus a full institutional implementation

This v1 notebook is a transparent repo-native diagnostic dashboard. It intentionally omits several data and modeling layers that would typically be required for a full institutional cross-asset implementation:

- No true positioning layer.
- No futures non-commercial positioning.
- No fund beta, survey, or option positioning inputs.
- No point-in-time macro support score.
- No broad 100+ cross-asset index universe.
- No Ornstein-Uhlenbeck residual model.
- No integrated residual model.
- No Lasso-based PC relevance selection.

Instead, 07b v1 uses the canonical 29-asset panel, weekly rolling PCA reconstruction residuals, trailing momentum, and existing `dominant_regime` labels as categorical context. The resulting scorecard should be interpreted as a transparent research diagnostic, not as a complete institutional signal stack.

In [ ]:
methodology_gap_table

## §10 Research interpretation and limitations

The latest scorecard is a diagnostic summary, not a signal. PCA residuals depend on the rolling window, the number of retained components, the complete-case sample, and the standardization convention. PCA signs and residual direction must be interpreted under the documented sign convention: positive means stronger than reconstruction, while negative means weaker than reconstruction.

Rolling PCA can be unstable in stress periods or when the cross-asset correlation structure changes. Momentum can confirm or contradict dislocation for valid reasons, including persistent trends, delayed mean reversion, or macro shocks. The existing `dominant_regime` labels are coarse context only and are not complete macro support scores. The latest scorecard date reflects the weekly resampling label; in this run, the week-ending label is 2026-05-01 while the available daily price history extends through 2026-04-30.

This notebook does not construct a portfolio, report strategy performance, make transaction-cost claims, or recommend trades.

In [ ]:
rich_assets = scorecard.sort_values("pca_dislocation_z", ascending=False).head(5)[["asset", "asset_class", "pca_dislocation_z", "momentum_score", "diagnostic_label"]]
cheap_assets = scorecard.sort_values("pca_dislocation_z", ascending=True).head(5)[["asset", "asset_class", "pca_dislocation_z", "momentum_score", "diagnostic_label"]]
label_counts = scorecard["diagnostic_label"].value_counts().rename_axis("diagnostic_label").reset_index(name="asset_count")

print(f"Latest scorecard date: {latest_date.date()}")
print("Top rich/strong assets versus PCA reconstruction:")
display(rich_assets)
print("Top cheap/weak assets versus PCA reconstruction:")
display(cheap_assets)
print("Diagnostic label counts:")
display(label_counts)

created_files = sorted(p.relative_to(ROOT).as_posix() for p in OUT_DIR.rglob("*") if p.is_file())
print("Notebook 07b artifacts:")
for path in created_files:
    print(f"- {path}")